In [ ]:
from pathlib import Path
import xarray as xr

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root. Start Jupyter from this repository or one of its subdirectories."
    )

REPO_ROOT = find_repo_root(Path.cwd())
RAW_NCAR_DIR = REPO_ROOT / "data" / "raw" / "ncar"
RAW_OISST_DIR = REPO_ROOT / "data" / "raw" / "oisst" / "v2.1"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
data_path = RAW_NCAR_DIR
flist = sorted(data_path.glob("uwnd.????.nc"))

if not flist:
    raise FileNotFoundError(f"No uwnd input files found in {data_path}")

output_file = PROCESSED_DIR / "uwnd_z850_jja_1991_2023.nc"


In [ ]:
ds = xr.open_mfdataset(flist)

In [ ]:
da = ds.uwnd

In [ ]:
da = da.sel(level=850.0)

In [ ]:
da = da.sel(time=slice("1991", "2023"))

In [ ]:
da = da.sel(time=da.time.dt.month.isin([6, 7, 8]))

In [ ]:
da = da.compute()

In [ ]:
da.to_netcdf(output_file)


In [ ]:
ds.close()

In [ ]:
da.close()